# RetailMind AI — Day 9: an LSTM that forecasts demand

How much will each store sell tomorrow? We read the last 28 days of a store's sales and predict the next day with an LSTM, and compare it with simple "no learning" baselines.

**The sales here are made up** (the real Kaggle Rossmann file needs a Kaggle account), so the scores show that the code works, not how well it would forecast real demand. Training takes about a minute, so this notebook **reads the saved results** from `python -m src.train_day9`.

In [1]:
# Work from the project root so the `src` package can be imported
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: day3-4


## 1. Look at the sales

In [2]:
from src.demand_data import load_sales, daily_table

table, source = load_sales()
wide = daily_table(table)
print(source)
print(f"{wide.shape[1]} stores x {wide.shape[0]} days: {wide.index[0].date()} to {wide.index[-1].date()}")

# one store, four weeks, as text bars (each # is about 250 sales). Sunday is closed.
print("\nStore 1, 28 days:")
for date, sales in wide[1].iloc[-28:].items():
    print(f"  {date.strftime('%a %d %b')}  {int(sales):>6}  " + "#" * int(sales // 250))

MADE-UP sales for 8 stores (the real Rossmann file was not found)
8 stores x 960 days: 2013-01-01 to 2015-08-18

Store 1, 28 days:
  Wed 22 Jul    4887  ###################
  Thu 23 Jul    4849  ###################
  Fri 24 Jul    4601  ##################
  Sat 25 Jul    6062  ########################
  Sun 26 Jul       0  
  Mon 27 Jul    5495  #####################
  Tue 28 Jul    4142  ################
  Wed 29 Jul    4037  ################
  Thu 30 Jul    4171  ################
  Fri 31 Jul    5034  ####################
  Sat 01 Aug    6189  ########################
  Sun 02 Aug       0  
  Mon 03 Aug    6113  ########################
  Tue 04 Aug    5079  ####################
  Wed 05 Aug    5221  ####################
  Thu 06 Aug    4256  #################
  Fri 07 Aug    4881  ###################
  Sat 08 Aug    5112  ####################
  Sun 09 Aug       0  
  Mon 10 Aug    5043  ####################
  Tue 11 Aug    4298  #################
  Wed 12 Aug    4030  ##############

## 2. How the windows are cut

Each example is the last 28 days (the question) and the next day (the answer), slid one day at a time. The split is **by time**: oldest days train, the next 8 weeks validate, the latest 8 weeks test.

In [3]:
from src.demand_data import split_windows
from src.train_day9 import LOOKBACK, HORIZON

train, validation, test = split_windows(wide, LOOKBACK, HORIZON)
print("windows  train:", len(train[0]), "| validation:", len(validation[0]), "| test:", len(test[0]))
print("one window: X has", train[0].shape[1], "days, y has", train[1].shape[1], "day")

windows  train: 6560 | validation: 448 | test: 448
one window: X has 28 days, y has 1 day


## 3. The results (from `python -m src.train_day9`)

In [4]:
import json
day9 = json.load(open("models/day9_results.json"))
print(day9["data"])
print(f"LSTM: {day9['weights']} weights, stopped after {day9['rounds_trained']} rounds\n")
print(f"{'Forecast of the next day':<26}{'MAE':>9}{'RMSE':>9}")
for name, s in day9["test_scores"].items():
    print(f"{name:<26}{s['mae']:>9}{s['rmse']:>9}")
print("\n" + day9["verdict"])

MADE-UP sales for 8 stores (the real Rossmann file was not found)
LSTM: 4385 weights, stopped after 69 rounds

Forecast of the next day        MAE     RMSE
LSTM                          248.3    329.9
same as yesterday            1991.1   3072.3
same as last week             665.2    853.1
same as 2 weeks ago           278.2    386.1

The LSTM beats the best baseline (same as 2 weeks ago) by 29.9 sales units of MAE.


## 4. Predicted vs actual (store 1, last 14 test days)

In [5]:
print(f"{'actual':>8}{'LSTM':>8}{'last week':>11}")
for row in day9["sample_last_14_days"]:
    print(f"{row['actual']:>8}{row['lstm']:>8}{row['same_as_last_week']:>11}")

  actual    LSTM  last week
    5221    4714       4037
    4256    4610       4171
    4881    5117       5034
    5112    5449       6189
       0       0          0
    5043    4971       6113
    4298    4406       5079
    4030    4114       5221
    4087    4357       4256
    5027    4861       4881
    5831    6400       5112
       0       0          0
    6047    5861       5043
    5222    5092       4298


## 5. Use the demand module

In [6]:
from src.demand_service import forecast_next_day

for store in list(wide.columns)[:3]:
    last_28_days = wide[store].to_numpy()[-LOOKBACK:]
    print("store", store, "-> forecast for tomorrow:", forecast_next_day(last_28_days))

store 1 -> forecast for tomorrow: 4554.4
store 2 -> forecast for tomorrow: 4528.0


store 3 -> forecast for tomorrow: 4113.9


## 6. Conclusion

- **The LSTM beat every baseline**, but only by 29.9 units of MAE (about 11%) over the strongest one, "same weekday 2 weeks ago". That baseline is the fair one here, because the made-up promotions repeat every 2 weeks. Against "yesterday" or "last week" the LSTM looks far better than it really is.
- **Not the real data.** The made-up sales have a clean repeating pattern. Real Rossmann sales are messier, so expect different numbers.
- **First training limit was too low:** the model hit the 40-round cap and was still improving. We raised the cap to 150 and let early stopping decide (it stopped at 69).
- One run, one seed.